In [0]:
from pyspark.sql import Row

dados = [
    Row(id=1, nome="João", idade=25),
    Row(id=2, nome="Maria", idade=30),
    Row(id=3, nome="Pedro", idade=35)
]

In [0]:
df = spark.createDataFrame(dados, ["id", "nome", "idade"])
display(df)

id,nome,idade
1,João,25
2,Maria,30
3,Pedro,35


In [0]:
df = spark.read.csv(
    "/Volumes/blueprint_spark/bronze/bronze/vendas.csv",
    header=True,
    inferSchema=True,
    sep=",",
    encoding="utf-8",
    nullValue="NA"
    )

In [0]:
display(df)

id,vendedor_id,vendedor,regiao,produto,valor,mes,data_venda
1,1,Ana,Sudeste,Notebook,3200.0,Jan,2026-01-05
2,1,Ana,Sudeste,Mouse,80.0,Jan,2026-01-07
3,2,Bruno,Sul,Monitor,950.0,Jan,2026-01-10
4,2,Bruno,Sul,Teclado,220.0,Fev,2026-02-02
5,3,Carla,Nordeste,Notebook,3400.0,Fev,2026-02-11
6,3,Carla,Nordeste,Webcam,150.0,Fev,2026-02-14
7,4,Diego,Centro-Oeste,Monitor,980.0,Mar,2026-03-03
8,4,Diego,Centro-Oeste,Notebook,3100.0,Mar,2026-03-09
9,5,Elaine,Norte,Mouse,75.0,Mar,2026-03-15
10,5,Elaine,Norte,Teclado,210.0,Mar,2026-03-20


In [0]:
df.printSchema()

root
 |-- id: integer (nullable = true)
 |-- vendedor_id: integer (nullable = true)
 |-- vendedor: string (nullable = true)
 |-- regiao: string (nullable = true)
 |-- produto: string (nullable = true)
 |-- valor: double (nullable = true)
 |-- mes: string (nullable = true)
 |-- data_venda: date (nullable = true)



In [0]:
df = spark.read.json(
    "/Volumes/blueprint_spark/bronze/bronze/vendas.json",
    multiLine=False,
    primitivesAsString=True,

    )


In [0]:
display(df)

data_venda,id,mes,produto,regiao,valor,vendedor,vendedor_id
2026-01-05,1,Jan,Notebook,Sudeste,3200.0,Ana,1
2026-01-07,2,Jan,Mouse,Sudeste,80.0,Ana,1
2026-01-10,3,Jan,Monitor,Sul,950.0,Bruno,2
2026-02-02,4,Fev,Teclado,Sul,220.0,Bruno,2
2026-02-11,5,Fev,Notebook,Nordeste,3400.0,Carla,3
2026-02-14,6,Fev,Webcam,Nordeste,150.0,Carla,3
2026-03-03,7,Mar,Monitor,Centro-Oeste,980.0,Diego,4
2026-03-09,8,Mar,Notebook,Centro-Oeste,3100.0,Diego,4
2026-03-15,9,Mar,Mouse,Norte,75.0,Elaine,5
2026-03-20,10,Mar,Teclado,Norte,210.0,Elaine,5


In [0]:
df = spark.read.parquet(
    '/Volumes/blueprint_spark/bronze/bronze/vendas.parquet',
    mergeSchema=True,
    )

display(df)

id,vendedor_id,vendedor,regiao,produto,valor,mes,data_venda
1,1,Ana,Sudeste,Notebook,3200.0,Jan,2026-01-05
2,1,Ana,Sudeste,Mouse,80.0,Jan,2026-01-07
3,2,Bruno,Sul,Monitor,950.0,Jan,2026-01-10
4,2,Bruno,Sul,Teclado,220.0,Fev,2026-02-02
5,3,Carla,Nordeste,Notebook,3400.0,Fev,2026-02-11
6,3,Carla,Nordeste,Webcam,150.0,Fev,2026-02-14
7,4,Diego,Centro-Oeste,Monitor,980.0,Mar,2026-03-03
8,4,Diego,Centro-Oeste,Notebook,3100.0,Mar,2026-03-09
9,5,Elaine,Norte,Mouse,75.0,Mar,2026-03-15
10,5,Elaine,Norte,Teclado,210.0,Mar,2026-03-20


In [0]:
df.columns

['id',
 'vendedor_id',
 'vendedor',
 'regiao',
 'produto',
 'valor',
 'mes',
 'data_venda']

In [0]:
['id',
 'vendedor_id',
 'vendedor',
 'regiao',
 'produto', 
 'valor',
 'mes',
 'data_venda'] 2024, 2025

In [0]:
['id',
 'vendedor_id',
 'vendedor',
 'regiao',
 'produto', 
 'valor',
 'mes',
 'data_venda',
 "Subcategoria_produto"
 ] 2026

In [0]:
df = spark.read.jdbc(
    url="jdbc:postgresql://dpg-dau2rto93c1s73cgql3g-a.oregon-postgres.render.com:5432/curso_spark",
    table="public.vendas_cidades",
    properties={
        "user": "curso_spark_user",
        "password": "Vz756WwUlxEdhYY8JJzGhSD974AZDPaI",
        "driver": "org.postgresql.Driver",
        "sslmode": "require",
    },
)

display(df)

id,cidade,produto,valor,data_venda
1,Salvador,Notebook,1191.52,2026-04-22
2,Recife,Webcam,2890.93,2026-07-27
3,Salvador,Monitor,1814.05,2026-08-11
4,Rio de Janeiro,Mouse,1881.74,2026-04-13
5,Salvador,Teclado,1156.95,2026-08-17
6,Fortaleza,Monitor,1194.74,2026-03-16
7,Recife,Monitor,1795.80,2026-04-17
8,Rio de Janeiro,Notebook,2930.43,2026-09-10
9,Salvador,Notebook,2041.59,2026-07-01
10,Salvador,Webcam,1168.22,2026-09-13


In [0]:
df.count()

15000

In [0]:
df.groupBy(spark_partition_id()).count().show()

+--------------------+-----+
|SPARK_PARTITION_ID()|count|
+--------------------+-----+
|                   0|15000|
+--------------------+-----+



In [0]:
df_numerico = spark.read.jdbc(
    url="jdbc:postgresql://dpg-dau2rto93c1s73cgql3g-a.oregon-postgres.render.com:5432/curso_spark",
    table="public.vendas_cidades",
    column="id",         # coluna numérica usada pra dividir a leitura
    lowerBound=1,         # menor valor de id na tabela
    upperBound=15000,      # maior valor de id na tabela
    numPartitions=4,      # em quantos pedaços dividir o intervalo (1 a 5000)
    properties={
        "user": "curso_spark_user",
        "password": "Vz756WwUlxEdhYY8JJzGhSD974AZDPaI",
        "driver": "org.postgresql.Driver",
        "sslmode": "require",
    },
)

# O Spark divide o intervalo 1-5000 em 4 pedaços iguais e abre 4 conexões ao mesmo tempo:
# partição 1: id entre 1 e 1250 | partição 2: id entre 1250 e 2500
# partição 3: id entre 2500 e 3750 | partição 4: id entre 3750 e 5000
df_numerico.groupBy(spark_partition_id()).count().show()

+--------------------+-----+
|SPARK_PARTITION_ID()|count|
+--------------------+-----+
|                   0| 3751|
|                   1| 3749|
|                   2| 3749|
|                   3| 3751|
+--------------------+-----+



In [0]:
df_cidade = spark.read.jdbc(
    url="jdbc:postgresql://dpg-dau2rto93c1s73cgql3g-a.oregon-postgres.render.com:5432/curso_spark",
    table="public.vendas_cidades",
    predicates=[
        "cidade = 'Salvador'",         # cada linha da lista = 1 conexão = 1 WHERE
        "cidade = 'São Paulo'",
        "cidade = 'Rio de Janeiro'",
        "cidade = 'Recife'",
        "cidade = 'Fortaleza'",
        "cidade = 'Curitiba'",
    ],
    properties={
        "user": "curso_spark_user",
        "password": "Vz756WwUlxEdhYY8JJzGhSD974AZDPaI",
        "driver": "org.postgresql.Driver",
        "sslmode": "require",
    },
)

# 6 predicados = 6 conexões abertas ao mesmo tempo = 6 partições, uma por cidade.
df_cidade.groupBy(spark_partition_id()).count().show()

+--------------------+-----+
|SPARK_PARTITION_ID()|count|
+--------------------+-----+
|                   0| 5935|
|                   1| 3094|
|                   2| 2239|
|                   3| 1796|
|                   4| 1197|
|                   5|  739|
+--------------------+-----+



In [0]:
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DateType,DoubleType

schema=StructType([
    StructField("id", IntegerType(), True),
    StructField("vendedor_id", IntegerType(), True),
    StructField("vendedor", StringType(), True),
    StructField("regiao", StringType(), True),
    StructField("produto", StringType(), True),
    StructField("valor", DoubleType(), True),
    StructField("mes", StringType(), True),
    StructField("data_venda", DateType(), True),
])

df = spark.read.csv(
    "/Volumes/blueprint_spark/bronze/bronze/vendas.csv",
    header=True,
    schema=schema
    )

display(df)

id,vendedor_id,vendedor,regiao,produto,valor,mes,data_venda
1,1,Ana,Sudeste,Notebook,3200.0,Jan,2026-01-05
2,1,Ana,Sudeste,Mouse,80.0,Jan,2026-01-07
3,2,Bruno,Sul,Monitor,950.0,Jan,2026-01-10
4,2,Bruno,Sul,Teclado,220.0,Fev,2026-02-02
5,3,Carla,Nordeste,Notebook,3400.0,Fev,2026-02-11
6,3,Carla,Nordeste,Webcam,150.0,Fev,2026-02-14
7,4,Diego,Centro-Oeste,Monitor,980.0,Mar,2026-03-03
8,4,Diego,Centro-Oeste,Notebook,3100.0,Mar,2026-03-09
9,5,Elaine,Norte,Mouse,75.0,Mar,2026-03-15
10,5,Elaine,Norte,Teclado,210.0,Mar,2026-03-20


In [0]:
df.show()

+---+-----------+--------+------------+--------+------+---+----------+
| id|vendedor_id|vendedor|      regiao| produto| valor|mes|data_venda|
+---+-----------+--------+------------+--------+------+---+----------+
|  1|          1|     Ana|     Sudeste|Notebook|3200.0|Jan|2026-01-05|
|  2|          1|     Ana|     Sudeste|   Mouse|  80.0|Jan|2026-01-07|
|  3|          2|   Bruno|         Sul| Monitor| 950.0|Jan|2026-01-10|
|  4|          2|   Bruno|         Sul| Teclado| 220.0|Fev|2026-02-02|
|  5|          3|   Carla|    Nordeste|Notebook|3400.0|Fev|2026-02-11|
|  6|          3|   Carla|    Nordeste|  Webcam| 150.0|Fev|2026-02-14|
|  7|          4|   Diego|Centro-Oeste| Monitor| 980.0|Mar|2026-03-03|
|  8|          4|   Diego|Centro-Oeste|Notebook|3100.0|Mar|2026-03-09|
|  9|          5|  Elaine|       Norte|   Mouse|  75.0|Mar|2026-03-15|
| 10|          5|  Elaine|       Norte| Teclado| 210.0|Mar|2026-03-20|
| 11|          1|     Ana|     Sudeste| Monitor| 970.0|Abr|2026-04-02|
| 12| 

In [0]:
display(df)

id,vendedor_id,vendedor,regiao,produto,valor,mes,data_venda
1,1,Ana,Sudeste,Notebook,3200.0,Jan,2026-01-05
2,1,Ana,Sudeste,Mouse,80.0,Jan,2026-01-07
3,2,Bruno,Sul,Monitor,950.0,Jan,2026-01-10
4,2,Bruno,Sul,Teclado,220.0,Fev,2026-02-02
5,3,Carla,Nordeste,Notebook,3400.0,Fev,2026-02-11
6,3,Carla,Nordeste,Webcam,150.0,Fev,2026-02-14
7,4,Diego,Centro-Oeste,Monitor,980.0,Mar,2026-03-03
8,4,Diego,Centro-Oeste,Notebook,3100.0,Mar,2026-03-09
9,5,Elaine,Norte,Mouse,75.0,Mar,2026-03-15
10,5,Elaine,Norte,Teclado,210.0,Mar,2026-03-20


In [0]:
df.printSchema()

root
 |-- id: integer (nullable = true)
 |-- vendedor_id: integer (nullable = true)
 |-- vendedor: string (nullable = true)
 |-- regiao: string (nullable = true)
 |-- produto: string (nullable = true)
 |-- valor: string (nullable = true)
 |-- mes: string (nullable = true)
 |-- data_venda: date (nullable = true)



In [0]:
df.columns

['id',
 'vendedor_id',
 'vendedor',
 'regiao',
 'produto',
 'valor',
 'mes',
 'data_venda']

In [0]:
df.dtypes

[('id', 'int'),
 ('vendedor_id', 'int'),
 ('vendedor', 'string'),
 ('regiao', 'string'),
 ('produto', 'string'),
 ('valor', 'string'),
 ('mes', 'string'),
 ('data_venda', 'date')]

In [0]:
df.count()

15

In [0]:
df.describe().show()

+-------+----------------+------------------+--------+------------+-------+------------------+----+
|summary|              id|       vendedor_id|vendedor|      regiao|produto|             valor| mes|
+-------+----------------+------------------+--------+------------+-------+------------------+----+
|  count|              15|                15|      15|          15|     15|                15|  15|
|   mean|             8.0|               3.0|    NULL|        NULL|   NULL|            1340.8|NULL|
| stddev|4.47213595499958|1.4638501094227998|    NULL|        NULL|   NULL|1435.6408325204463|NULL|
|    min|               1|                 1|     Ana|Centro-Oeste|Monitor|              75.0| Abr|
|    max|              15|                 5|  Elaine|         Sul| Webcam|            3400.0| Mar|
+-------+----------------+------------------+--------+------------+-------+------------------+----+



In [0]:
df.summary().show()

+-------+----------------+------------------+--------+------------+-------+------------------+----+
|summary|              id|       vendedor_id|vendedor|      regiao|produto|             valor| mes|
+-------+----------------+------------------+--------+------------+-------+------------------+----+
|  count|              15|                15|      15|          15|     15|                15|  15|
|   mean|             8.0|               3.0|    NULL|        NULL|   NULL|            1340.8|NULL|
| stddev|4.47213595499958|1.4638501094227998|    NULL|        NULL|   NULL|1435.6408325204463|NULL|
|    min|               1|                 1|     Ana|Centro-Oeste|Monitor|              75.0| Abr|
|    25%|               4|                 2|    NULL|        NULL|   NULL|             145.0|NULL|
|    50%|               8|                 3|    NULL|        NULL|   NULL|             950.0|NULL|
|    75%|              12|                 4|    NULL|        NULL|   NULL|            3200.0|NULL|


In [0]:
df.limit(5).show()

+---+-----------+--------+--------+--------+------+---+----------+
| id|vendedor_id|vendedor|  regiao| produto| valor|mes|data_venda|
+---+-----------+--------+--------+--------+------+---+----------+
|  1|          1|     Ana| Sudeste|Notebook|3200.0|Jan|2026-01-05|
|  2|          1|     Ana| Sudeste|   Mouse|  80.0|Jan|2026-01-07|
|  3|          2|   Bruno|     Sul| Monitor| 950.0|Jan|2026-01-10|
|  4|          2|   Bruno|     Sul| Teclado| 220.0|Fev|2026-02-02|
|  5|          3|   Carla|Nordeste|Notebook|3400.0|Fev|2026-02-11|
+---+-----------+--------+--------+--------+------+---+----------+

